# Step 1B — Magic Camera

The child holds up a real object to the camera. The photo is sent to the backend as a JPEG.

```
Character selection
    └─ Camera photo (JPEG)
           ├─ POST /api/sketch-preview
           │      ├─ 1. Label extraction   (Flash Lite reads the photo → 3–6 word label)
           │      ├─ 2. Safety check       (SAFE / UNSAFE)
           │      └─ 3. Illustration gen   (Gemini image model → storybook art)
           └─ POST /api/story-opening      (opening text + first image, using prop label)
```

**Set `IMAGE_PATH` below to point at any photo on your machine.**

---
## Setup

In [ ]:
import sys, os, base64
sys.path.insert(0, '../backend')

from dotenv import load_dotenv
load_dotenv('../backend/.env')

print('PROJECT:      ', os.environ.get('GOOGLE_CLOUD_PROJECT', '❌ not set'))
print('IMAGE_MODEL:  ', os.environ.get('IMAGE_MODEL', '❌ not set'))
print('GEMINI_API_KEY:', 'set' if os.environ.get('GEMINI_API_KEY') else '❌ not set')

---
## Initial Step — Character Selection

In [ ]:
from characters import CHARACTERS, get_character

print(f'{len(CHARACTERS)} characters available:\n')
for char in CHARACTERS.values():
    print(f'  {char.id:20s}  lang={char.language:10s}  voice={char.voice_name:15s}  {char.name}')

In [ ]:
# Pick a character
char = get_character('grandma-rose')

print(f'Name:         {char.name}')
print(f'Language:     {char.language}')
print(f'Voice:        {char.voice_name}')
print(f'Image style:  {char.image_style}')
print(f'\n--- System prompt (first 400 chars) ---')
print(char.system_prompt[:400], '...')

---
## Load Camera Photo

Point `IMAGE_PATH` at any JPEG or PNG on your machine — a photo of a toy, a book, food, anything.

In [ ]:
# ← Set this to your image
IMAGE_PATH = '/path/to/your/photo.jpg'

from IPython.display import Image, display
from PIL import Image as PILImage
import io

# Load and convert to JPEG (camera output is always JPEG in the app)
pil_img = PILImage.open(IMAGE_PATH).convert('RGB')
pil_img.thumbnail((640, 480))   # downscale to match camera resolution

buf = io.BytesIO()
pil_img.save(buf, format='JPEG', quality=85)
photo_bytes = buf.getvalue()
photo_b64 = base64.b64encode(photo_bytes).decode()

print(f'Photo: {pil_img.size[0]}x{pil_img.size[1]}px, {len(photo_bytes):,} bytes')
display(Image(data=photo_bytes))

---
## Step 1 — Label Extraction

Flash Lite receives the photo and outputs a short child-friendly label for whatever it sees.

In [ ]:
from google.genai import types
from image_gen import EXTRACT_MODEL, _extract_client

print(f'Model: {EXTRACT_MODEL}\n')

label_response = await _extract_client.aio.models.generate_content(
    model=EXTRACT_MODEL,
    contents=[
        types.Part(inline_data=types.Blob(mime_type='image/jpeg', data=photo_bytes)),
        types.Part(text=(
            'A young child held this up to the camera. In 3–6 simple, friendly words describe '
            'the main subject — e.g. \'a friendly blue dragon\', \'a big rainbow castle\', '
            '\'a red toy car\'. No punctuation at the end. Keep it imaginative and warm.'
        )),
    ],
)

label = label_response.text.strip().rstrip('.')
print(f'Label: {label!r}')

---
## Step 2 — Safety Check

The label is screened before any image is generated.

In [ ]:
from image_gen import _is_safe_for_children

safe = await _is_safe_for_children(label)

print(f'Label: {label!r}')
print(f'Safe:  {safe}')

if not safe:
    print('\n❌ Would be rejected — endpoint returns HTTP 400 unsafe_content')
else:
    print('\n✅ Proceeding to illustration')

---
## Step 3 — Illustration Generation

The photo + label go to the Gemini image model. It recreates the real object as warm storybook art.

The label is the crucial hint — it tells the model exactly what the child sees, so the output matches the prop.

In [ ]:
from image_gen import IMAGE_MODEL, _generate_from_sketch

print(f'Image model: {IMAGE_MODEL}')
print(f'Art style:   {char.image_style}')
print(f'Label hint:  {label!r}')
print()

image_b64, mime_type = await _generate_from_sketch(photo_bytes, char.image_style, label)

print(f'Generated: {mime_type}, {len(image_b64):,} chars')
display(Image(data=base64.b64decode(image_b64)))

---
## Full `POST /api/sketch-preview`

Runs all three steps as one call — exactly what the frontend sends.

In [ ]:
from image_gen import sketch_preview, SketchPreviewRequest

request = SketchPreviewRequest(
    sketch_data=photo_b64,
    image_style=char.image_style,
)

result = await sketch_preview(request)

print(f'Label: {result.label!r}')
print(f'Image: {result.mime_type}, {len(result.image_data):,} chars')
display(Image(data=base64.b64decode(result.image_data)))

---
## `POST /api/story-opening` — Opening Text + First Illustration

The prop label (`result.label`) is passed as `prop_description` so the opening story puts the real-world object into the narrative.

The image model generates the first story scene — the prop reimagined inside the story world.

In [ ]:
from image_gen import _generate_opening

prop_description = result.label

print(f'Character:  {char.name}')
print(f'Prop:       {prop_description!r}')
print()

opening_text, scene_description = await _generate_opening(
    character_name=char.name,
    character_language=char.language,
    theme='camera_prop',
    prop_description=prop_description,
)

print('=== STORY OPENING ===')
print(opening_text)
print('\n=== SCENE DESCRIPTION (for image gen) ===')
print(scene_description)

In [ ]:
from image_gen import generate_story_opening, StoryOpeningRequest

opening_request = StoryOpeningRequest(
    character_id=char.id,
    character_name=char.name,
    character_language=char.language,
    image_style=char.image_style,
    theme='camera_prop',
    prop_description=result.label,
)

opening_result = await generate_story_opening(opening_request)

print('Opening text:')
print(opening_result.opening_text)
print(f'\nScene: {opening_result.scene_description}')
print(f'Image: {opening_result.mime_type}, {len(opening_result.image_data):,} chars')
display(Image(data=base64.b64decode(opening_result.image_data)))